# qtest in 10 minutes

**qtest** is a pytest-native testing library for quantum circuits. It gives you assertions that understand sampling noise, fidelity, and unitary equivalence -- so you can write tests for quantum code the same way you write tests for classical code.

This quickstart walks through four scenes:

1. Verify a Bell state distribution (`assert_distribution_close`)
2. Verify a state vector (`assert_state_close`)
3. Verify unitarity (`assert_unitary`)
4. Verify two circuits implement the same unitary (`assert_circuit_equivalent`)

Run the cells in order. The histograms render with matplotlib; the assertions either pass silently or raise `AssertionError` with a detailed diff.

> **Prereqs:** `pip install qtest qiskit-aer matplotlib`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit

from qtest.assertions import (
    assert_circuit_equivalent,
    assert_distribution_close,
    assert_state_close,
    assert_unitary,
)
from qtest.backends.registry import get_default_backend

## 1. The Bell state, statistically

A Bell pair is the canonical entangled state. Applying `H` on qubit 0 then `CNOT(0 -> 1)` to `|00>` should yield a 50/50 mix of `00` and `11`, with `01` and `10` impossible.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()
bell.draw('mpl')

In [ ]:
# Run 4_000 shots and plot the empirical distribution
counts = get_default_backend().run_circuit(bell, shots=4000, seed=42)
labels = ['00', '01', '10', '11']
values = [counts.get(l, 0) for l in labels]

plt.bar(labels, values)
plt.ylabel('counts (4000 shots)')
plt.title('Bell state measurement outcomes')
plt.show()

In [ ]:
# qtest's distribution assertion takes care of the statistics for us
assert_distribution_close(
    bell,
    expected={'00': 0.5, '11': 0.5},
    tolerance=0.05,
    shots=4000,
    seed=42,
)
print('First test green. You just verified a quantum algorithm!')

## 2. The |+> state, exactly

When you don't want to add measurements, `assert_state_close` compares the *state vector* directly. It uses fidelity by default, so a global phase factor never spuriously fails the test.

In [ ]:
plus = QuantumCircuit(1)
plus.h(0)  # no measurement -- we want the state, not samples

assert_state_close(plus, expected_state='plus', tolerance=1e-9)
print('|+> verified to 1e-9 fidelity.')

## 3. Is my custom gate even unitary?

`assert_unitary` accepts numpy matrices, Qiskit `Gate` objects, or whole circuits. Useful when you're hand-building rotation matrices or composing gates from a paper.

In [ ]:
theta = 0.7
ry = np.array([
    [np.cos(theta / 2), -np.sin(theta / 2)],
    [np.sin(theta / 2),  np.cos(theta / 2)],
], dtype=complex)

assert_unitary(ry, tolerance=1e-12)
print('RY({:.2f}) is unitary.'.format(theta))

## 4. Two circuits, same unitary

When refactoring a circuit (or testing a transpiler), you want to confirm the *operator* didn't change. `assert_circuit_equivalent` uses process fidelity, which is phase-insensitive.

In [ ]:
direct = QuantumCircuit(1)
direct.h(0)

# H = RY(pi/2) * Z
decomposed = QuantumCircuit(1)
decomposed.z(0)
decomposed.ry(np.pi / 2, 0)

assert_circuit_equivalent(direct, decomposed, tolerance=1e-9)
print('Decomposition matches H exactly.')

## What next?

- **`examples/01_basic_assertions.py`** -- the four assertions in pytest form
- **`examples/02_property_based_testing.py`** -- combine qtest with Hypothesis
- **`examples/03_pytest_plugin.py`** -- CLI flags, markers, `--qtest-summary`
- **`examples/04_testing_grover.py`** -- a complete Grover test suite
- **`examples/05_testing_transpiler.py`** -- regression tests for circuit rewrites

And the full reference docs: <https://qtest.readthedocs.io>.